# BSM L07G — Pochodzenie aplikacji, zaufanie w runtime i higiena kopii zapasowych (Android)

## Tryb pracy
To nie jest lab z Pythonem. Implementujesz rozwiązania w **Android Studio / Kotlin** w starterze projektu `lesson_g_app`.
Ten notebook służy jako:
- instrukcja krok-po-kroku (co otworzyć, gdzie kliknąć, czego szukać w kodzie),
- formularz odpowiedzi,
- mechanizm wysyłki odpowiedzi do backendu.

## Starter projektu
W tym repozytorium używasz folderu:
- `student/apps/lesson_g_app`

## Jak powstają odpowiedzi (ważne)
- **Zadanie 1 (G01)**: odpowiedź jest wysyłana automatycznie z aplikacji (nie ma w notebooku komórki „Wyślij”).
- **Zadania 2-4 (G02-G04)**: odpowiedzi wysyłasz z notebooka (komórki „Formularz odpowiedzi”).

## Literatura i dokumentacja (na ten lab)
Obowiązkowo odwołuj się do dokumentacji platformy (linki poniżej) i uzasadniaj decyzje bezpieczeństwa.

- Android Developers: App signing (overview)
- Android Developers: Data and file storage (SharedPreferences, internal storage)
- Android Developers: Auto Backup / Backup rules (w tym `android:allowBackup`, `android:fullBackupContent`, `android:dataExtractionRules`)
- Android Developers: `EncryptedSharedPreferences`, `MasterKey` (AndroidX Security Crypto)
- OWASP MASVS / MSTG: sekcje o przechowywaniu sekretów, uprawnieniach i prywatności

Z sylabusa: temat labu podpina się pod „Cechy bezpieczeństwa platformy Android” i praktykę „stosowania technik bezpieczeństwa zgodnie z dokumentacją deweloperską”.


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("
", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)


# G01 — Manifest i audyt prywatności (odpowiedź wysyła aplikacja)

## Cel
Zobaczyć w praktyce, że deklaracja w `AndroidManifest.xml` i rzeczywiste użycie funkcji to dwie różne rzeczy.
W prawdziwych review (np. aplikacje sklepowe, audyty prywatności) patrzy się na:
- listę uprawnień,
- powód biznesowy i techniczny,
- moment prośby o uprawnienie (runtime vs manifest),
- minimalizację zakresu (principle of least privilege).

## Co masz zrobić
1. Otwórz projekt `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik manifestu: `app/src/main/AndroidManifest.xml`.
1. Zrób mapę: dla każdego `<uses-permission ...>` zapisz, która funkcja aplikacji go realnie potrzebuje:
- lokalizacja: mapka (pobranie bieżącej lokalizacji)
- kamera: zrobienie zdjęcia
- galeria: wybór zdjęcia
- internet: pobranie kafelka mapy z usługi zewnętrznej
1. Odpowiedz sobie na pytania kontrolne (nie wysyłasz ich do backendu, ale są kluczowe do zrozumienia):
- Czy wszystkie zadeklarowane uprawnienia są faktycznie potrzebne? Jeśli tak, to w jakiej sytuacji?
- Które z nich są „runtime permissions” i kiedy aplikacja powinna o nie prosić?
- Czy aplikacja ma sensowny fallback, jeśli użytkownik odmówi?

## Jak zaliczasz (automatycznie)
To zadanie jest powiązane z aplikacją.
1. Uruchom aplikację na emulatorze lub urządzeniu.
1. Wpisz swoje **Student ID** w ekranie aplikacji (sekcja „Student / Task 1”).
1. Kliknij „Request permissions” i przejdź cały przepływ.
1. Jeśli wszystko jest poprawnie, aplikacja sama wykona wysyłkę dla `G01`.

Uwaga: w tym notebooku **nie ma** komórki „Wyślij” dla G01.


# G02 — Bezpieczne przechowywanie klucza API (secure storage) + minimalizacja wycieków

## Kontekst teoretyczny
Sekret (np. API key do usługi map) to typowy element, który wycieka przez:
- przypadkowe wklejenie do repozytorium,
- logi (`Log.d`), crash reporting,
- backup/migrację (Auto Backup),
- pliki zasobów w APK.

W tym zadaniu masz zaprojektować minimalny, ale sensowny przepływ:
- gdzie sekret jest trzymany,
- jak jest pobierany przez UI,
- jak zapewniasz, że nie ląduje „przy okazji” w miejscach niekontrolowanych.

## Co masz zrobić (krok po kroku)
1. Otwórz `student/apps/lesson_g_app` w Android Studio.
1. Przeczytaj kod mapy w: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
   Szukaj stałej `MAP_API_KEY_PREF` i miejsca, gdzie przekazywany jest `mapApiKey` do `ApiMapCard(...)`.
1. Zlokalizuj warstwę bezpiecznych preferencji:
- `app/src/main/java/com/example/secretlab/secure/SecurePrefs.kt`
- (prawdopodobnie) `EncryptedSharedPreferences` + `MasterKey`
1. Zaprojektuj i zaimplementuj brakujący fragment przepływu tak, żeby:
- klucz API był zapisywany/odczytywany z bezpiecznego storage,
- aplikacja nie działała „na skróty” (np. nie hardcode’uj klucza w `MainActivity.kt`),
- w razie braku klucza UI pokazywał sensowną informację (jest już komunikat „Map API key missing.”).

Wskazówki projektowe (nie są kodem rozwiązania):
- jeśli w UI dodajesz pole tekstowe do wpisania klucza, zadbaj o minimalizację ryzyka: nie loguj wartości, nie trzymaj jej w stanie dłużej niż potrzeba.
- API key traktuj jak sekret: ogranicz ekspozycję w kodzie i w narzędziach.

## Dokumentacja (linki do czytania)
- AndroidX Security Crypto: `EncryptedSharedPreferences`, `MasterKey`
- Android Developers: „Store data securely” / „Data and file storage”

## Jak zdobyć kod zaliczeniowy
1. Uruchom testy unit:
- Android Studio: Gradle tool window -> `:app` -> `verification` -> `testDebugUnitTest`
1. Albo uruchom zadanie evidence (z katalogu projektu):
- `./gradlew :app:bsmEvidence`
1. Jeżeli G02 jest poprawne, w konsoli zobaczysz krótki kod (5 znaków). To jest odpowiedź do wysłania.


In [ ]:
#@title G02 — Formularz odpowiedzi
# Wklej 5-znakowy kod z `:app:bsmEvidence`.
code_g02 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g02.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G02", final_answer)


# G03 — Żądanie do backendu bramkowane integralnością (runtime trust)

## Kontekst teoretyczny
„Nazwa paczki” i „zainstalowana aplikacja” to za mało, żeby ufać klientowi.
W praktyce backend często wymaga sygnału integralności/proweniencji, np.:
- czy aplikacja pochodzi z zaufanego kanału,
- czy nie jest repackowana/tamperowana,
- czy urządzenie nie jest w stanie, który łamie model bezpieczeństwa.

Nawet jeśli w tym labie backend jest „mockowany”, chodzi o poprawne ułożenie:
- sprawdzenia tożsamości aplikacji,
- powiązania żądania z tą tożsamością (binding),
- bezpiecznego fallbacku, gdy nie da się ustalić zaufania.

## Co masz zrobić (krok po kroku)
1. Otwórz plik: `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`.
   Zobacz, jakie warunki muszą być spełnione, żeby pojawił się kod dla zadania 3.
1. Przejrzyj miejsce, w którym aplikacja wykonuje sieciowe wysyłki:
- `app/src/main/java/com/example/secretlab/MainActivity.kt`
  Szukaj `submitAnswer(...)` i tego, jakie dane są wysyłane.
1. Zidentyfikuj, czego brakuje, aby żądanie było „związane” z tożsamością aplikacji.
   W praktyce oznacza to, że serwer powinien móc zweryfikować, że:
- żądanie pochodzi z oczekiwanej aplikacji (podpis / certyfikat / identyfikator),
- nie jest łatwe do podrobienia przez inną aplikację.

Uwaga: Nie implementujesz tutaj pełnego Play Integrity API (to wymagałoby kluczy i konfiguracji).
W labie celem jest minimalny, testowalny mechanizm, który modeluje takie wiązanie.

## Jak zdobyć kod zaliczeniowy
1. Uruchom testy unit albo `./gradlew :app:bsmEvidence`.
1. Jeżeli G03 jest poprawne, w konsoli zobaczysz 5-znakowy kod. To jest odpowiedź do wysłania.


In [ ]:
#@title G03 — Formularz odpowiedzi
# Wklej 5-znakowy kod z `:app:bsmEvidence`.
code_g03 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g03.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G03", final_answer)


# G04 — Higiena sekretów przy backupie i migracji

## Kontekst teoretyczny
Android może przenosić stan aplikacji między urządzeniami (Auto Backup / Backup & restore).
To bywa wygodne, ale dla sekretów może być ryzykowne, bo:
- sekret „device-bound” nie powinien migrować,
- tokeny/sesje nie powinny odtwarzać się bez ponownego uwierzytelnienia,
- dane debugowe nie powinny trafiać do backupu.

W tym zadaniu chodzi o świadome ustawienie polityki backupu:
- co wolno kopiować,
- czego nie wolno,
- co trzeba odtworzyć po reinstalacji w bezpieczny sposób.

## Co masz zrobić (krok po kroku)
1. Otwórz manifest: `app/src/main/AndroidManifest.xml`.
   Zwróć uwagę na `android:allowBackup` (w starterze jest ustawione na `true`).
1. Zdecyduj i uzasadnij politykę:
- czy aplikacja ma w ogóle pozwalać na backup,
- jeśli tak, to jakie pliki/prefsy wykluczasz (sekrety, klucze, tokeny).
1. Jeśli używasz reguł backupu:
- dodaj odpowiedni plik XML do `app/src/main/res/xml/` (np. rules dla backupu),
- podepnij go w manifeście przez właściwy atrybut (zależnie od wersji Androida).
1. Upewnij się, że sekret labowy dla G04 nie jest trwale migrowany w niepożądany sposób.

## Skąd wziąć odpowiedź
W aplikacji istnieje 5-znakowa wartość, która jest zapisana w „ścieżce sekretów” używanej w tym labie.
Po wdrożeniu polityki backupu/migracji i higieny sekretów odczytaj tę wartość (zgodnie z instrukcją w kodzie) i wklej ją poniżej.

Wskazówka: szukaj stałych w `MainActivity.kt` związanych z zadaniem 4 (`TASK_4_...`).


In [ ]:
#@title G04 — Formularz odpowiedzi
# Wklej 5-znakową wartość sekretu dla zadania 4.
secret_g04 = ""  #@param {type:"string"}

final_answer = prepare_answer(secret_g04.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G04", final_answer)
